In [5]:
import jams
import numpy as np
import os
import pretty_midi
import random
np.int = int # deprecated np.int
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
from collections import defaultdict
import math

In [6]:
@dataclass
class TabNote:
    """Represents a single note in tablature"""
    time: float
    string: int  # 1-6 (1 = high e, 6 = low E)
    fret: int
    duration: float = 0.25  # quarter note by default
    techniques: List[str] = None
    
    def __post_init__(self):
        if self.techniques is None:
            self.techniques = []


# Extended technique symbols for realistic tablature
TECHNIQUE_SYMBOLS = {
    'hammer_on': 'h',
    'pull_off': 'p',
    'slide_up': '/',
    'slide_down': '\\',
    'bend': 'b',
    'release': 'r',
    'vibrato': '~',
    'palm_mute': 'PM',
    'harmonic': '<>',
    'tapping': 't',
    'tremolo_pick': 'tr'
}

# Standard tuning
STRING_NAMES = ['e', 'B', 'G', 'D', 'A', 'E']

In [7]:
class TabFormatter:
    """Handles the visual formatting of guitar tablature"""
    
    def __init__(self, measures_per_line=4, tempo=120):
        self.measures_per_line = measures_per_line
        self.tempo = tempo
        self.char_per_beat = 4  # characters allocated per beat
        self.measure_width = self.char_per_beat * 4  # 4/4 time
        
    def calculate_position(self, time: float, measure_start_time: float) -> int:
        """Calculate character position within a measure based on time"""
        beats_from_start = (time - measure_start_time) * (self.tempo / 60)
        return int(beats_from_start * self.char_per_beat)
    
    def create_measure_template(self) -> List[str]:
        """Create an empty measure for all 6 strings"""
        width = self.measure_width
        return ['-' * width for _ in range(6)]
    
    def format_fret(self, fret: int, techniques: List[str]) -> str:
        """Format fret number with technique symbols"""
        fret_str = str(fret)
        
        # Add technique symbols
        tech_str = ''
        for tech in techniques:
            symbol = TECHNIQUE_SYMBOLS.get(tech, '')
            if symbol and len(symbol) <= 2:
                tech_str += symbol
        
        return fret_str + tech_str
    
    def add_note_to_measure(self, measure: List[str], note: TabNote, 
                           position: int, next_position: int = None) -> List[str]:
        """Add a note to the measure at the specified position"""
        string_idx = 6 - note.string  # Convert to 0-indexed from bottom
        fret_text = self.format_fret(note.fret, note.techniques)
        
        # Calculate how much space we need
        note_width = len(fret_text)
        
        # Ensure position is within bounds
        if position >= len(measure[string_idx]):
            position = len(measure[string_idx]) - note_width
        
        # Convert measure string to list for easier manipulation
        line = list(measure[string_idx])
        
        # Place the note
        for i, char in enumerate(fret_text):
            if position + i < len(line):
                line[position + i] = char
        
        # Fill with dashes after the note if there's space
        if next_position:
            for i in range(position + note_width, min(next_position, len(line))):
                if line[i] not in '0123456789hpbr/\\~t<>':
                    line[i] = '-'
        
        measure[string_idx] = ''.join(line)
        return measure
    
    def add_bar_line(self, measures: List[List[str]], is_final: bool = False) -> List[str]:
        """Add bar lines to separate measures"""
        bar = '|' if not is_final else '|'
        return [line + bar for line in measures]
    
    def create_tab_line(self, measures: List[List[str]], line_num: int) -> str:
        """Create a complete line of tablature with string names"""
        output = []
        
        # Add string labels
        for string_idx, string_name in enumerate(STRING_NAMES):
            line_parts = [f"{string_name}|"]
            
            for measure in measures:
                line_parts.append(measure[string_idx])
            
            # Add final bar line
            line_parts.append('|')
            output.append(''.join(line_parts))
        
        return '\n'.join(output)
    
    def add_rhythm_notation(self, measures: List[List[str]], notes_by_measure: List[List[TabNote]]) -> str:
        """Add rhythm notation above the tablature"""
        rhythm_line = ' ' * 3  # Space for string label
        
        for measure_idx, measure_notes in enumerate(notes_by_measure):
            measure_rhythm = [' '] * self.measure_width
            
            for note in measure_notes:
                pos = self.calculate_position(note.time, measure_idx * 4.0)
                if pos < len(measure_rhythm):
                    # Add rhythm symbols based on duration
                    if note.duration >= 1.0:
                        measure_rhythm[pos] = 'w'  # whole note
                    elif note.duration >= 0.5:
                        measure_rhythm[pos] = 'h'  # half note
                    elif note.duration >= 0.25:
                        measure_rhythm[pos] = 'q'  # quarter note
                    elif note.duration >= 0.125:
                        measure_rhythm[pos] = 'e'  # eighth note
                    else:
                        measure_rhythm[pos] = 's'  # sixteenth note
            
            rhythm_line += '|' + ''.join(measure_rhythm)
        
        rhythm_line += '|'
        return rhythm_line

In [37]:
def jam_to_tab(jam, measures_per_line=4, tempo=120, time_signature=(4, 4)) -> str:
    """
    Convert a JAMS object into professional-looking guitar tablature.
    
    Args:
        jam: JAMS object containing tab_note annotations
        measures_per_line: Number of measures to display per line
        tempo: Tempo in BPM (used for spacing calculations)
        time_signature: Tuple of (beats_per_measure, beat_unit)
    
    Returns:
        String containing formatted guitar tablature
    """
    
    # Extract notes from JAMS object
    notes = []
    for ann in jam.annotations:
        if ann.namespace == "tab_note":
            for obs in ann.data:
                val = obs.value
                duration = obs.duration if hasattr(obs, 'duration') else 0.25
                
                notes.append(TabNote(
                    time=obs.time,
                    string=val['string'],
                    fret=val['fret'],
                    duration=duration,
                    techniques=val.get('techniques', [])
                ))
    
    if not notes:
        return "No tablature data found in JAMS object."
    
    # Sort notes by time
    notes.sort(key=lambda x: x.time)
    
    # Initialize formatter
    formatter = TabFormatter(measures_per_line=measures_per_line, tempo=tempo)
    
    # Group notes by measure
    beats_per_measure = time_signature[0]
    beat_duration = 60.0 / tempo  # seconds per beat
    measure_duration = beats_per_measure * beat_duration
    
    max_time = notes[-1].time + notes[-1].duration
    total_measures = math.ceil(max_time / measure_duration)
    
    # Organize notes by measure
    notes_by_measure = [[] for _ in range(total_measures)]
    for note in notes:
        measure_idx = int(note.time / measure_duration)
        if measure_idx < total_measures:
            notes_by_measure[measure_idx].append(note)
    
    # Build tablature output
    output_lines = []
    
    # Add title and tempo marking
    output_lines.append("=" * 60)
    output_lines.append(f"Tempo: {tempo} BPM | Time: {time_signature[0]}/{time_signature[1]}")
    output_lines.append("=" * 60)
    output_lines.append("")
    
    # Process measures in groups (lines)
    for line_start in range(0, total_measures, measures_per_line):
        line_end = min(line_start + measures_per_line, total_measures)
        line_measures = []
        line_notes = []
        
        for measure_idx in range(line_start, line_end):
            measure = formatter.create_measure_template()
            measure_notes = notes_by_measure[measure_idx]
            measure_notes.sort(key=lambda x: x.time)
            
            # Add each note to the measure
            for i, note in enumerate(measure_notes):
                measure_start_time = measure_idx * measure_duration
                position = formatter.calculate_position(note.time, measure_start_time)
                
                # Calculate next position for proper spacing
                next_position = None
                if i + 1 < len(measure_notes):
                    next_position = formatter.calculate_position(
                        measure_notes[i + 1].time, measure_start_time
                    )
                
                measure = formatter.add_note_to_measure(
                    measure, note, position, next_position
                )
            
            line_measures.append(measure)
            line_notes.append(measure_notes)
        
        # Add rhythm notation (optional, commented out for cleaner look)
        # rhythm = formatter.add_rhythm_notation(line_measures, line_notes)
        # output_lines.append(rhythm)
        
        # Create the tablature line
        tab_line = formatter.create_tab_line(line_measures, line_start // measures_per_line)
        output_lines.append(tab_line)
        output_lines.append("")
    
    # Add legend for techniques
    output_lines.append("=" * 60)
    output_lines.append("Technique Legend:")
    output_lines.append("h = hammer-on  |  p = pull-off  |  b = bend  |  r = release")
    output_lines.append("/ = slide up   |  \\ = slide down |  ~ = vibrato  |  t = tapping")
    output_lines.append("PM = palm mute |  <> = harmonic")
    output_lines.append("=" * 60)
    
    return '\n'.join(output_lines)

In [38]:
# test with a sample jams
jam = jams.JAMS()
ann = jams.Annotation(namespace="tab_note")

ann.append(time=0.0, duration=0.5, value={'string': 1, 'fret': 3, 'techniques': ['bend']})
ann.append(time=0.5, duration=0.5, value={'string': 2, 'fret': 5, 'techniques': ['hammer on', 'vibrato']})
ann.append(time=1.0, duration=0.5, value={'string': 6, 'fret': 0, 'techniques': []})

jam.annotations.append(ann)

In [39]:
tab_output = jam_to_tab(jam, measures_per_line=4, tempo=120)
print(tab_output)

Tempo: 120 BPM | Time: 4/4

e|--------0-------|
B|----------------|
G|----------------|
D|----------------|
A|----5~----------|
E|3b--------------|

Technique Legend:
h = hammer-on  |  p = pull-off  |  b = bend  |  r = release
/ = slide up   |  \ = slide down |  ~ = vibrato  |  t = tapping
PM = palm mute |  <> = harmonic


In [11]:
midi_dir= '/data/akshaj/MusicAI/GuitarSet/MIDIAnnotations/'

In [12]:
def midi_to_jams(midi_path):
    pm = pretty_midi.PrettyMIDI(midi_path)
    guitar_notes = pm.instruments[0].notes  # first instrument
    jam = jams.JAMS()

    note_ann = jams.Annotation(namespace='note')  # create a note annotation
    for note in guitar_notes:
        note_ann.append(
            time=note.start,
            duration=note.end - note.start,
            value=note.pitch,
            confidence=note.velocity / 127  # normalize velocity
        )

    jam.annotations.append(note_ann)
    return jam

In [14]:
def encode_notes_for_test(jam):
    '''adds sample random expressive techniques, random strings, and frets for testing/tab output purposes'''
    new_ann = jams.Annotation(namespace='tab_note')
    tech_options = ["slide", "vibrato", "hammer-on", "pull-off", "bend", None]

    # iterate over existing note 
    note_ann = jam.annotations[0]  
    for obs in note_ann.data:
        pitch = obs.value  # the original pitch

        value = {
            "pitch": pitch,
            "string": random.randint(1, 6),       # random string 1-6
            "fret": random.randint(0, 12),        # random fret 0-12
            "techniques": [random.choice(tech_options)] if random.random() < 0.5 else []
        }

        new_ann.append(time=obs.time, duration=obs.duration, value=value, confidence=obs.confidence)

    # add new tab_note annotation
    jam.annotations.append(new_ann)
    return jam

In [40]:
example_midi = os.path.join(midi_dir, '05_Funk2-108-Eb_solo.mid')

jam = midi_to_jams(example_midi)
jam = encode_notes_for_test(jam)
tab_output = jam_to_tab(jam, measures_per_line=4, tempo=108)
print(tab_output)

Tempo: 108 BPM | Time: 4/4

e|------0~-----------------------------7---56------11-------------|
B|----------5~----------------------------------------------------|
G|----3--------1--------------------------------------------------|
D|--------------5------------------------91-----10----------------|
A|-------------------------------------------9--------------------|
E|-------103-2---9-----------------------------5------------------|

e|-------------------------------3--------------------------------|
B|-------111~------------2b---0------------12--------------------1|
G|------------------------3--------------------2---------------5--|
D|--------12~-----------------------------------------------------|
A|----------------------------------------------------------------|
E|-----------12------------7-------------------5-6-----4----------|

e|------11---------------------------------------------7b-2-------|
B|--------------------------9--------------------------------11---|
G|----------------

In [22]:
import lilypond
from music21 import stream, note, chord, tempo, metadata
from music21 import tablature as m21tab

In [24]:
def jams_to_lilypond(jam_file, output_pdf='output.pdf'):
    """
    Convert JAMS to PDF tablature using music21 + LilyPond
    """
    jam = jams.load(jam_file)
    
    # Create music21 score
    score = stream.Score()
    guitar_part = stream.Part()
    guitar_part.insert(0, m21tab.GuitarFretBoard())
    
    # Add tempo
    guitar_part.insert(0, tempo.MetronomeMark(number=120))
    
    # Convert JAMS notes to music21
    for ann in jam.annotations:
        if ann.namespace == "tab_note":
            for obs in ann.data:
                val = obs.value
                
                # Create note with tablature info
                n = note.Note()
                n.quarterLength = obs.duration * 4
                
                # Add fret information
                fret_note = m21tab.FretNote(
                    string=val['string'],
                    fret=val['fret']
                )
                
                # Handle techniques
                if 'techniques' in val and val['techniques']:
                    for tech in val['techniques']:
                        if tech == 'hammer_on':
                            n.articulations.append(articulations.Accent())
                        # Add more technique mappings
                
                guitar_part.insert(obs.time, n)
    
    score.append(guitar_part)
    
    # Render to PDF using LilyPond
    score.write('lily.pdf', fp=output_pdf)
    
    return output_pdf


In [30]:
from bulletproof_converter import midi_to_jams_fixed, encode_notes_for_tab, jams_to_notation_simple

example_midi = os.path.join(midi_dir, '05_Funk2-108-Eb_solo.mid')

jam = midi_to_jams_fixed(example_midi)
jam = encode_notes_for_tab(jam)

# Simple version - all quarter notes (always works!)
jams_to_notation_simple(jam, 'output.xml')

# OR quantized version - better rhythm
# jams_to_notation_quantized(jam, 'output.xml', quantize_to='eighth')

Converting 61 notes...
✓ Created output.xml


'output.xml'

In [36]:
from real_tab_converter import complete_midi_to_tab_workflow
from musicxml_converter import musicxml_to_pdf
import os

# Your MIDI file
midi_file = os.path.join(midi_dir, '05_Funk2-108-Eb_solo.mid')

# Convert to tablature (intelligent mapping!)
complete_midi_to_tab_workflow(midi_file, 'tablature.xml', tempo_bpm=108)

MIDI to Tablature - Complete Workflow

1. Converting MIDI to tablature...
Converted 61 notes to tablature
Pitch range: 56 - 75

2. Generating MusicXML...
Converting 61 notes to MusicXML...
✓ Created tablature.xml

✓ Complete!
Output: tablature.xml

Next steps:
1. Open in MuseScore: musescore tablature.xml
2. Export to PDF: File → Export → PDF


'tablature.xml'

In [53]:
# Test to see if we can get epxressive techniques into the tablature
from real_tab_converter import midi_to_jams_with_tablature, jams_to_musicxml_real
import pretty_midi

# Test midi file
midi_path = os.path.join(midi_dir, '05_Funk2-108-Eb_solo.mid')

# Startup code
pm = pretty_midi.PrettyMIDI(midi_path)
# Try to get tempo from MIDI
tempo_changes = pm.get_tempo_changes()
if len(tempo_changes[1]) > 0:
    tempo_bpm = int(tempo_changes[1][0])
    print(f"\nDetected tempo: {tempo_bpm} BPM")
else:
    tempo_bpm = 120
    print(f"\nUsing default tempo: {tempo_bpm} BPM")

# Create jams object out of midi data
jam = midi_to_jams_with_tablature(midi_path)

# Populate jams object with random expressive techniques
jam = encode_notes_for_test(jam)

# Convert to MusicXML and ouptut file
output_file = jams_to_musicxml_real(jam, 'techniques.xml', tempo_bpm)


Detected tempo: 60 BPM
Converted 61 notes to tablature
Pitch range: 56 - 75
Converting 61 notes to MusicXML...
✓ Created techniques.xml
